# CryptoLab 12 — From Shannon to Modern Security

**CYBR 3570 Cryptography**

This lab moves from the one-time pad and Shannon-style perfect secrecy to modern adversarial security games. The goal is **not** to write formal proofs. Instead, you will use small experiments to see what different security claims actually mean.

> **Important:** Several constructions in this notebook use tiny parameters or simplified mathematics so the behavior is visible. They are teaching models, not production cryptography.

## Learning Objectives

By the end of this notebook, you should be able to:

- explain the difference between perfect secrecy and computational security;
- demonstrate why a correctly used one-time pad hides the plaintext;
- demonstrate why OTP key reuse is catastrophic;
- model an indistinguishability-style security game;
- explain why deterministic encryption fails under repeated/chosen plaintexts;
- compare deterministic and randomized encryption behavior;
- model the idea behind ElGamal randomized public-key encryption;
- demonstrate ciphertext malleability and explain why authenticated encryption matters;
- write a simple security claim in terms of a goal, adversary, and assumptions.

In [1]:
import os
import random
import secrets
from collections import Counter
from typing import Callable

from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print("Notebook ready.")

Notebook ready.


# Part 1 — Perfect Secrecy in a Tiny World

For a one-time pad, encryption is:

```text
C = M XOR K
```

If the key is uniformly random, the same length as the message, and used exactly once, every ciphertext is compatible with every plaintext through exactly one key.

In [2]:
def xor_int(a: int, b: int) -> int:
    return a ^ b

messages = list(range(4))   # 2-bit messages: 00, 01, 10, 11
keys = list(range(4))       # 2-bit keys

# Show all possible encryptions.
for m in messages:
    row = [xor_int(m, k) for k in keys]
    print(f"M={m:02b} ->", [f"{c:02b}" for c in row])

M=00 -> ['00', '01', '10', '11']
M=01 -> ['01', '00', '11', '10']
M=10 -> ['10', '11', '00', '01']
M=11 -> ['11', '10', '01', '00']


### Exercise 1 — What does the ciphertext tell you?

Choose any ciphertext value from `00` through `11`. For each possible plaintext, determine the unique key that would produce that ciphertext.

Then answer:

1. After observing the ciphertext, can the attacker rule out any plaintext?
- No, all possible options are avaliable
2. Why does this illustrate perfect secrecy?
- Knowing the ciphertext tells me nothing about the message

In [5]:
observed_c = 0b10

# TODO: Fill this dictionary with plaintext -> required key.
possible_explanations = {}

for m in messages:
    required_key = xor_int(m, observed_c)
    print(required_key)

possible_explanations

2
3
0
1


{}

# Part 2 — The One-Time Pad Fails When the Key Is Reused

If the same pad encrypts two messages:

```text
C1 = M1 XOR K
C2 = M2 XOR K
```

then:

```text
C1 XOR C2 = M1 XOR M2
```

The key cancels out.

In [6]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))

m1 = b"ATTACK AT DAWN!!"
m2 = b"RETREAT AT NOON!"
key = secrets.token_bytes(len(m1))

c1 = xor_bytes(m1, key)
c2 = xor_bytes(m2, key)

print("C1 XOR C2:", xor_bytes(c1, c2).hex())
print("M1 XOR M2:", xor_bytes(m1, m2).hex())
print("Equal?", xor_bytes(c1, c2) == xor_bytes(m1, m2))

C1 XOR C2: 13110013060a74611574640f18016f00
M1 XOR M2: 13110013060a74611574640f18016f00
Equal? True


### Exercise 2 — Known plaintext recovery

Assume an attacker learns `m1` and has captured both `c1` and `c2`. Recover `m2` without directly using the original `key` variable.

Explain why this attack does **not** contradict the proof of one-time-pad security.
- Because one time pads can only be used once, this just proves that they cannot be used twice becasue it will reveal the message. 

In [7]:
# TODO: Recover the reused pad from m1 and c1, then recover m2.
recovered_key = xor_bytes(m1, c1)
recovered_m2 = xor_bytes(recovered_key, c2)

print("Recovered message:", recovered_m2)

Recovered message: b'RETREAT AT NOON!'


# Part 3 — From Perfect Secrecy to Computational Security

Perfect secrecy says the attacker learns nothing even with unlimited computation. Modern cryptography usually makes a weaker but much more practical claim: attacks may exist in principle, but no **efficient** attacker should succeed with meaningful probability.

The work factor grows exponentially with key size for an ideal brute-force search.

In [8]:
for bits in [40, 56, 80, 128, 256]:
    keyspace = 2 ** bits
    print(f"{bits:3d}-bit key -> {keyspace:.3e} possible keys")

 40-bit key -> 1.100e+12 possible keys
 56-bit key -> 7.206e+16 possible keys
 80-bit key -> 1.209e+24 possible keys
128-bit key -> 3.403e+38 possible keys
256-bit key -> 1.158e+77 possible keys


### Exercise 3 — Security is a claim with a budget

Suppose an attacker can test `10^12` keys per second. Estimate the time to search **half** of a 56-bit, 80-bit, and 128-bit key space. Express your answers in years.

What does this tell you about the phrase “computationally infeasible”?

In [10]:
tests_per_second = 10**12
seconds_per_year = 365.25 * 24 * 60 * 60

# TODO: Calculate expected brute-force years for each key size.
estimates = {}
for bits in [56, 80, 128, 256]:
    keyspace = 2 ** bits
    years = keyspace / (tests_per_second * seconds_per_year)
    estimates[bits] = years

estimates

{56: 0.0022833673675415095,
 80: 38308.547532595294,
 128: 1.0782897524556319e+19,
 256: 3.669229891921952e+57}

# Part 4 — An Indistinguishability Game

A security game makes the attacker's task explicit. In a left-or-right game:

1. The attacker chooses equal-length messages `M0` and `M1`.
2. The challenger secretly chooses a bit `b`.
3. The challenger encrypts `Mb`.
4. The attacker guesses `b`.

Random guessing succeeds about 50% of the time.

In [11]:
def left_or_right_game(encrypt: Callable[[bytes], bytes], adversary, rounds=1000):
    wins = 0
    for _ in range(rounds):
        m0, m1 = adversary.choose_messages()
        if len(m0) != len(m1):
            raise ValueError("Messages must have the same length")
        b = secrets.randbelow(2)
        c = encrypt(m0 if b == 0 else m1)
        guess = adversary.guess(c)
        wins += int(guess == b)
    return wins / rounds

class RandomGuessAdversary:
    def choose_messages(self):
        return b"PAY=0000", b"PAY=9999"
    def guess(self, ciphertext):
        return secrets.randbelow(2)

### Exercise 4 — Establish a baseline

Create an encryption function that returns random bytes of the same length as its input. Run the game with `RandomGuessAdversary`.

You should get a success rate near 0.5. Why is 0.5 the baseline rather than 0?

In [ ]:
def random_world_encrypt(message: bytes) -> bytes:
    # TODO: Return uniformly random bytes with the same length as message.
    return bytes(len(message))

# TODO: Run left_or_right_game(...) and record the result.


# Part 5 — Deterministic Encryption Leaks Equality

A deterministic encryption function always produces the same ciphertext for the same key and plaintext. Even if the ciphertext is unreadable, repeated plaintexts produce repeated ciphertexts.

In [12]:
# A toy deterministic "encryption" based on a fixed secret permutation of byte values.
perm = list(range(256))
random.Random(3570).shuffle(perm)

def deterministic_toy_encrypt(message: bytes) -> bytes:
    return bytes(perm[b] for b in message)

for message in [b"YES", b"NO!", b"YES"]:
    print(message, deterministic_toy_encrypt(message).hex())

b'YES' 74d529
b'NO!' 45de0f
b'YES' 74d529


### Exercise 5 — Build a chosen-plaintext distinguisher

An attacker is allowed to ask for an encryption of `M0` before receiving the challenge ciphertext. Use that capability to distinguish whether the challenge encrypted `M0` or `M1` under deterministic encryption.

Write down the attack in plain English, then implement the comparison below.

In [17]:
m0 = b"PAY=0000"
m1 = b"PAY=9999"
reference = deterministic_toy_encrypt(m0)

# Simulate a challenge.
b = secrets.randbelow(2)
challenge = deterministic_toy_encrypt(m0 if b == 0 else m1)

# TODO: Guess 0 if challenge equals the known encryption of m0; otherwise guess 1.
guess = 0 if challenge == reference else 1
print("Secret bit:", b, "Guess:", guess)

Secret bit: 1 Guess: 1


# Part 6 — Randomization Breaks Equality Patterns

Modern encryption schemes normally use fresh randomness or a unique nonce so that encrypting the same plaintext twice does not produce the same ciphertext.

In [18]:
aes_key = AESGCM.generate_key(bit_length=128)
aesgcm = AESGCM(aes_key)

def randomized_encrypt(message: bytes) -> bytes:
    nonce = os.urandom(12)
    return nonce + aesgcm.encrypt(nonce, message, None)

for _ in range(3):
    print(randomized_encrypt(b"PAY=0000").hex())

cb8b8125da7b25ae13aeb636ba47f9ee36e2cd4aab42ea1bc47286ea71750b26545e16bd
a2058766b5fd1d77891f4eaaa85770ace3528d4c08ac684bca6f55a6893bb81ff2ba21a0
f174d81ddef8fba11923554e5c74962ba43ba8131dd24bdf8ea881fa19d130ed2e9a375e


### Exercise 6 — Compare observable patterns

Encrypt the same message 100 times with `deterministic_toy_encrypt` and 100 times with `randomized_encrypt`. Count the number of distinct ciphertexts in each set.

Explain what equality information is exposed in each case.

In [19]:
# TODO: Build both ciphertext sets and count distinct values.
deterministic_outputs = [ ]
randomized_outputs = []

for i in range (100):
    deterministic_outputs.append(deterministic_toy_encrypt(b"hello"))
    randomized_outputs.append(randomized_encrypt(b"hello"))
print("Distinct deterministic:", len(set(deterministic_outputs)))
print("Distinct randomized:", len(set(randomized_outputs)))

Distinct deterministic: 1
Distinct randomized: 100


# Part 7 — Toy ElGamal: Randomized Public-Key Encryption

ElGamal is closely related to Diffie–Hellman. We will use tiny numbers to make the algebra visible.

For public parameters `(p, g)`:

```text
private key: x
public key:  y = g^x mod p

Encrypt message m using fresh random k:
c1 = g^k mod p
c2 = m * y^k mod p

Decrypt:
m = c2 * (c1^x)^(-1) mod p
```

Real implementations use carefully selected groups and message encodings. This tiny version is for learning only.

In [20]:
p = 467
g = 2
x = 127
y = pow(g, x, p)

def elgamal_encrypt(m: int):
    if not 1 <= m < p:
        raise ValueError("Message must be a nonzero group element")
    k = secrets.randbelow(p - 2) + 1
    c1 = pow(g, k, p)
    shared = pow(y, k, p)
    c2 = (m * shared) % p
    return c1, c2

def elgamal_decrypt(ciphertext):
    c1, c2 = ciphertext
    shared = pow(c1, x, p)
    shared_inv = pow(shared, -1, p)
    return (c2 * shared_inv) % p

message = 123
for _ in range(3):
    ct = elgamal_encrypt(message)
    print(ct, "->", elgamal_decrypt(ct))

(46, 347) -> 123
(69, 119) -> 123
(282, 61) -> 123


### Exercise 7 — Why does the same plaintext look different?

Encrypt the same integer 20 times. Count the number of distinct `(c1, c2)` pairs.

Then explain which value provides the per-encryption randomness and how this resembles ephemeral Diffie–Hellman.

In [22]:
# TODO: Collect 20 ElGamal ciphertexts for the same message.
ciphertexts = []
for _ in range(20):
    ciphertexts.append(elgamal_encrypt(123))
print("Distinct ciphertexts:", len(set(ciphertexts)))

Distinct ciphertexts: 20


# Part 8 — Malleability: Confidentiality Is Not Integrity

A stream-style encryption that XORs plaintext with a keystream can hide content while still allowing predictable modification. If an attacker flips a bit in the ciphertext, the corresponding plaintext bit flips after decryption.

In [23]:
plain = bytearray(b"admin=0;amount=100")
stream_key = secrets.token_bytes(len(plain))
cipher = bytearray(xor_bytes(bytes(plain), stream_key))

# Flip the ASCII bit that changes '0' (0x30) to '1' (0x31).
index = plain.index(ord("0"))
cipher[index] ^= 0x01

tampered_plain = xor_bytes(bytes(cipher), stream_key)
print(tampered_plain)

b'admin=1;amount=100'


### Exercise 8 — Compare with authenticated encryption

Use AES-GCM to encrypt `b"admin=0;amount=100"`. Flip one bit in the ciphertext and attempt to decrypt it.

What happens? Why is this failure behavior a security feature?

In [28]:
message = b"admin=0;amount=100"
nonce = os.urandom(12)
protected = bytearray(aesgcm.encrypt(nonce, message, None))

index = plain.index(ord("0"))
protected[index] ^= 0x01
try:
    tampered_plain = aesgcm.decrypt(nonce, protected, None)
except Exception:
    print("Tampering detected.")


Tampering detected.


# Part 9 — Write the Security Claim

A useful cryptographic claim should identify at least three things:

1. **Goal** — What property are we trying to preserve?
2. **Adversary** — What can the attacker observe, choose, modify, or compromise?
3. **Assumptions** — What must remain true for the claim to hold?

### Exercise 9 — Secure notes scenario

Your final-project team is building a secure notes application. Notes are encrypted with AES-GCM using a key derived from the user's master password.

Write a concise threat-model statement that addresses:

- the confidentiality goal; 
    - an attacker can see a cipher text and not be able to get the original plain text
- the integrity goal; 
    - prove the original sender; prove that the message has not been modified 
- what the attacker can read or modify;
    - read the cipher text, they can make changes to the cipher text 
- what the attacker does **not** initially know;
    - plain text, master password
- at least two things the system does **not** protect against.
    - Nonce reuse, insecure implementation, weak password too small key space

# Final Reflection

Answer in 1–3 sentences each.

1. Why is perfect secrecy stronger than the security we usually expect from AES?
- Perfect secrecy is stronger because it is mathematically impossible to break the secrecy. The usual AES makes it hard to brute force it becasue there is so much length to break through. 
2. Why is key reuse fatal for a one-time pad but normal for a modern multi-use encryption scheme?
- A key reuse is fatal for a one time pad is becasue it is only meant to be a one time thing. If the key is reused then it reveals the whole message. In the mordern encryption there are other factors that go into decryption so key reuse is not fatal. 
3. What does an indistinguishability game force us to specify about an attacker?
- A security game makes the attacker's task explicit. 1. The attacker chooses equal-length messages `M0` and `M1`. The challenger secretly chooses a bit `b`. The challenger encrypts `Mb`. The attacker guesses `b`. Random guessing succeeds about 50% of the time.
4. Why is randomized or nonce-based encryption usually preferable to deterministic encryption?
- Randomized encryption usually preferable becasue it makes sure that all the different encryption is not the same. In just basic deterministic encryption no matter how many encryption is done they are all the same. The randomized makes sure that every single encryption is different from one another. 
5. Why does authenticated encryption represent a stronger system building block than confidentiality-only encryption?
- Authenticated encryption is a stronger building block because it encorporated the full CIA triad. Confidentiality-only encryption makes sure that it hides the data from the eavesdropper. It does not check if the cipher text has been tampered with. 

---
## References

- Mike Rosulek, *The Joy of Cryptography*, Chapter 1: https://joyofcryptography.com/otp/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 4: https://joyofcryptography.com/comp/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 8: https://joyofcryptography.com/cpa/
- Mike Rosulek, *The Joy of Cryptography*, Chapter 9, *Chosen-Ciphertext Attacks Against Encryption*.

**Do not use the toy OTP, substitution, or ElGamal constructions in this notebook to protect real data.**